# Elevator Environment

In this Jupyter Notebook, we are going to learn how the elevator environment works.

## Setup

### Using Google Colab

If you are using Google Colab (and we encourage you to do so), please run the following code cell. If you are not using Google Colab, you can skip this code cell.

**Note**: The path `'/content/drive/'` cannot be changed. For example, if your assignment folder in Google Drive is located at `My Drive -> CSXX46A2-Student`, you should specify the path as `'/content/drive/MyDrive/CSXX46A2-Student'`.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
except ImportError:
    print("Local Jupyter environment detected; Google Drive was not mounted.")
else:
    drive.mount("/content/drive")

PACKAGE_DIR = Path("/content/drive/MyDrive/CSXX46A2-Student")
if not (PACKAGE_DIR / "selfDefinedEnvs").is_dir():
    PACKAGE_DIR = Path.cwd()
if not (PACKAGE_DIR / "selfDefinedEnvs").is_dir():
    raise FileNotFoundError(
        "Extract the Colab ZIP into MyDrive or run Jupyter from its extracted folder."
    )
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))
print(f"Assignment folder: {PACKAGE_DIR}")


### Installing Dependencies

The elevator task is implemented using the `PyRDDLGym` library. Before we begin, please install the following packages.

**Note**: If you are using Google Colab, you may need to restart the session. Please follow the prompt to do so.

In [ ]:
!pip -q install "pyRDDLGym==2.7" "rddlrepository==2.2"


## Environment

The Elevator environment models evening rush hours when people from different floors in a building want to go down to the bottom floor using elevators.

The building has 5 floors and 1 elevator. Each floor may have several people waiting. The objective is to pick up passengers from various floors and deliver them to the first floor. New passengers may arrive at each floor while the elevator is in operation. The elevator can move up and down, and pick up and drop off passengers. However, it can only do so when the door is open, and it can only move when the door is closed.

In [ ]:
import pyRDDLGym
from pyRDDLGym.core.env import RDDLEnv

import numpy as np

import gymnasium as gym
from gymnasium.spaces import Box, Discrete
from gymnasium.wrappers import RecordEpisodeStatistics

from utils import DictToListWrapper

### Initialization

To initialize the environment, call the `Elevator` class. Here, we will use the `DictToListWrapper` to convert the environment's state from a dictionary to a list, with the detail given in the "Environment Description" section below.

In [ ]:
def create_elevator_env():
    # env = pyRDDLGym.make('Elevators', instance_id)
    env = RDDLEnv(
        domain=str(PACKAGE_DIR / "selfDefinedEnvs" / "domain.rddl"),
        instance=str(PACKAGE_DIR / "selfDefinedEnvs" / "instance5.rddl"),  # instance-5 file
    )
    # If your observation is a Dict of booleans, flatten it:
    env = DictToListWrapper(env)
    env = RecordEpisodeStatistics(env)
    return env

env = create_elevator_env().env

### Observation and Action Space

Using the following code cell, we can check the observation and action space of the environment, with the detailed descriptions.

In [ ]:
print(f"Observation space: {env.observation_space}")
env.get_state_description()

print(f"Action space: {env.action_space}")
env.get_action_description()

### Interaction

The agent interacts with the environment following the [OpenAI Gym API](https://gymnasium.farama.org/). The environment provides the following methods:

- `reset()`: Resets the environment and returns the initial state along with any additional information (usually empty).
- `step(action)`: Takes an action in the environment and returns:
    - *next state*: The resulting state after the action.
    - *reward*: The reward received for the action.
    - *done*: A boolean indicating whether the episode has ended.
    - *truncated*: A boolean indicating whether the episode was truncated (terminated for any unspecified reason), which occurs when the 200-step time limit is reached.
    - *info*: Additional information returned as a dictionary.
- `close()`: Closes the environment and releases any resources.

A template for interacting with the environment is shown below:

In [ ]:
state, info = env.reset()

for i in range(20):
    # randomly sample an action from the action space
    action = env.action_space.sample()

    print(f"Action: {action}")

    next_state, reward, terminated, truncated, info = env.step(action)

    print(f"Next state: {next_state}")
    print(f"Reward: {reward}")
    print(f"Terminated: {terminated}")
    print(f"Truncated: {truncated}")

    done = terminated or truncated
    if done:
        state, info = env.reset()
